# FundSmart Triage Service Evaluation

This notebook evaluates the Docker-hosted `triage_service` as a black-box HTTP service.

It compares `v1` and `v2` by calling:

- `POST /triage?version=v1`
- `POST /triage?version=v2`

The notebook does not import pipeline internals. It can evaluate either combined benchmark JSONL, such as `sythetic_tests/synthetic_generated.jsonl`, or split files, `sythetic_tests/synthetic_complaints.jsonl` plus `sythetic_tests/gold_labels.jsonl`. It scores triage quality, metadata extraction, signal coverage, and deterministic acknowledgement safety. Ragas is optional and is used only as an LLM judge for acknowledgement groundedness, coherence, and compliance.

## Expected Docker Setup

Start the service from the repo root before running the notebook:

```bash
docker compose -f infra/docker-compose.yml up --build
```

By default the API is exposed at `http://localhost:8001`.

In [ ]:
# Parameters. Papermill overrides these with -P from scripts/run_triage_service_evaluation.sh.
TRIAGE_SERVICE_URL = "http://localhost:8001"
TRIAGE_TEST_DATA_FORMAT = "auto"
TRIAGE_TEST_DATA = ""
TRIAGE_COMPLAINTS_DATA = ""
TRIAGE_GOLD_LABELS = ""
TRIAGE_EVAL_TIMEOUT_SECONDS = "60"
RUN_RAGAS_EVAL = "false"
EVALUATION_OUTPUT_DIR = ""


In [ ]:
from __future__ import annotations

import json
import os
import re
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path
from typing import Any

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "services" / "triage_service").exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()


def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


def parameter_value(name: str, env_name: str, default: str) -> str:
    value = globals().get(name)
    if value is None or str(value).strip() == "":
        return os.getenv(env_name, default)
    return str(value)


load_env_file(REPO_ROOT / ".env")
load_env_file(REPO_ROOT / "services" / "triage_service" / ".env")

SERVICE_BASE_URL = parameter_value("TRIAGE_SERVICE_URL", "TRIAGE_SERVICE_URL", "http://localhost:8001").rstrip("/")
TEST_DATA_FORMAT = parameter_value("TRIAGE_TEST_DATA_FORMAT", "TRIAGE_TEST_DATA_FORMAT", "auto").strip().lower()
TEST_DATA_PATH = Path(
    parameter_value("TRIAGE_TEST_DATA", "TRIAGE_TEST_DATA", str(REPO_ROOT / "sythetic_tests" / "synthetic_generated.jsonl"))
).resolve()
COMPLAINTS_DATA_PATH = Path(
    parameter_value("TRIAGE_COMPLAINTS_DATA", "TRIAGE_COMPLAINTS_DATA", str(REPO_ROOT / "sythetic_tests" / "synthetic_complaints.jsonl"))
).resolve()
GOLD_LABELS_PATH = Path(
    parameter_value("TRIAGE_GOLD_LABELS", "TRIAGE_GOLD_LABELS", str(REPO_ROOT / "sythetic_tests" / "gold_labels.jsonl"))
).resolve()
API_KEY = os.getenv("TRIAGE_SERVICE_API_KEY", "").strip()
VERSIONS = ["v1", "v2"]
REQUEST_TIMEOUT_SECONDS = int(parameter_value("TRIAGE_EVAL_TIMEOUT_SECONDS", "TRIAGE_EVAL_TIMEOUT_SECONDS", "60"))
RUN_RAGAS_EVAL_PARAM = parameter_value("RUN_RAGAS_EVAL", "RUN_RAGAS_EVAL", "false")
EVALUATION_OUTPUT_DIR_PATH = Path(
    parameter_value("EVALUATION_OUTPUT_DIR", "EVALUATION_OUTPUT_DIR", str(REPO_ROOT / "evaluation" / "results" / "triage_service"))
).resolve()

if TEST_DATA_FORMAT not in {"auto", "combined", "split"}:
    raise ValueError("TRIAGE_TEST_DATA_FORMAT must be auto, combined, or split")

print(f"Service URL:       {SERVICE_BASE_URL}")
print(f"Test data format:  {TEST_DATA_FORMAT}")
print(f"Combined data:     {TEST_DATA_PATH}")
print(f"Complaints data:   {COMPLAINTS_DATA_PATH}")
print(f"Gold labels data:  {GOLD_LABELS_PATH}")
print(f"Versions:          {VERSIONS}")
print(f"Output dir:        {EVALUATION_OUTPUT_DIR_PATH}")


In [ ]:
def request_json(method: str, path: str, payload: dict[str, Any] | None = None) -> dict[str, Any]:
    url = f"{SERVICE_BASE_URL}{path}"
    headers = {"Accept": "application/json"}
    data = None
    if payload is not None:
        headers["Content-Type"] = "application/json"
        data = json.dumps(payload).encode("utf-8")
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"
    request = urllib.request.Request(url, data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"HTTP {exc.code} from {url}: {body}") from exc
    except urllib.error.URLError as exc:
        raise RuntimeError(
            f"Could not reach triage service at {SERVICE_BASE_URL}. "
            "Start it with: docker compose -f infra/docker-compose.yml up --build"
        ) from exc

health = request_json("GET", "/health")
health

In [ ]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for line_no, line in enumerate(path.read_text().splitlines(), start=1):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(f"Invalid JSONL at {path}:{line_no}: {exc}") from exc
    return rows


def metadata_from_split_complaint(row: dict[str, Any]) -> dict[str, Any]:
    return {
        "channel": row.get("channel"),
        "received": row.get("received"),
        "customer_id": row.get("customer_id"),
        "subject": row.get("subject"),
        "thread_context": row.get("thread_context"),
        "agent": row.get("agent"),
        "duration": row.get("duration"),
        "note": row.get("note"),
    }


def complaint_document_from_split_complaint(row: dict[str, Any]) -> str:
    header_labels = [
        ("Channel", row.get("channel")),
        ("Received", row.get("received")),
        ("Customer ID", row.get("customer_id")),
        ("Subject", row.get("subject")),
        ("Thread context", row.get("thread_context")),
        ("Agent", row.get("agent")),
        ("Duration", row.get("duration")),
        ("Note", row.get("note")),
    ]
    headers = [f"**{label}:** {value}" for label, value in header_labels if value is not None]
    message = (row.get("message") or "").strip()
    return "\n".join(headers) + f"\n\n```\n{message}\n```"


def load_combined_cases(path: Path) -> list[dict[str, Any]]:
    cases = load_jsonl(path)
    for case in cases:
        if "complaint_document" not in case:
            raise ValueError(f"Combined case {case.get('id')} is missing complaint_document")
    return cases


def load_split_cases(complaints_path: Path, labels_path: Path) -> list[dict[str, Any]]:
    complaints = load_jsonl(complaints_path)
    labels = load_jsonl(labels_path)
    labels_by_id = {row["id"]: row for row in labels}
    missing_labels = [row["id"] for row in complaints if row["id"] not in labels_by_id]
    extra_labels = sorted(set(labels_by_id) - {row["id"] for row in complaints})
    if missing_labels:
        raise ValueError(f"Missing gold labels for complaint ids: {missing_labels}")
    if extra_labels:
        raise ValueError(f"Gold labels without matching complaints: {extra_labels}")

    cases: list[dict[str, Any]] = []
    for complaint in complaints:
        label = labels_by_id[complaint["id"]]
        cases.append(
            {
                "id": complaint["id"],
                "source": "synthetic",
                "scenario_type": label.get("scenario_type") or complaint.get("scenario_type"),
                "complaint_document": complaint_document_from_split_complaint(complaint),
                "metadata": metadata_from_split_complaint(complaint),
                "expected_category": label["expected_category"],
                "expected_severity": label["expected_severity"],
                "expected_routing": label["expected_routing"],
                "expected_sla": label.get("expected_sla"),
                "scenario_tags": label.get("scenario_tags") or [],
                "expected_signals": label.get("expected_signals") or [],
                "expected_preferences": label.get("expected_preferences") or [],
                "forbidden_signals": label.get("forbidden_signals") or [],
                "must_detect": label.get("must_detect") or [],
                "must_not_detect": label.get("must_not_detect") or [],
                "customer_preferences": label.get("customer_preferences") or [],
            }
        )
    return cases


def load_cases() -> tuple[list[dict[str, Any]], str]:
    if TEST_DATA_FORMAT == "combined":
        return load_combined_cases(TEST_DATA_PATH), "combined"
    if TEST_DATA_FORMAT == "split":
        return load_split_cases(COMPLAINTS_DATA_PATH, GOLD_LABELS_PATH), "split"
    if "TRIAGE_TEST_DATA" in os.environ:
        return load_combined_cases(TEST_DATA_PATH), "combined"
    if COMPLAINTS_DATA_PATH.exists() and GOLD_LABELS_PATH.exists():
        return load_split_cases(COMPLAINTS_DATA_PATH, GOLD_LABELS_PATH), "split"
    return load_combined_cases(TEST_DATA_PATH), "combined"


cases, resolved_data_format = load_cases()
print(f"Loaded {len(cases)} test cases from {resolved_data_format} data")
pd.DataFrame(
    {
        "id": case["id"],
        "source": case.get("source"),
        "scenario_type": case.get("scenario_type"),
        "expected_category": case.get("expected_category"),
        "expected_severity": case.get("expected_severity"),
        "expected_routing": case.get("expected_routing"),
        "expected_sla": case.get("expected_sla"),
    }
    for case in cases
)


## Run v1 and v2 Through the Docker Service

Each row is sent as the API payload. The LLM input inside the service remains `complaint_document`; labels are only used here for benchmark scoring.

In [ ]:
def payload_for(case: dict[str, Any]) -> dict[str, Any]:
    return {
        "id": case["id"],
        "complaint_document": case["complaint_document"],
        "source": case.get("source"),
        "scenario_type": case.get("scenario_type"),
        "metadata": case.get("metadata") or {},
    }


def call_triage(case: dict[str, Any], version: str) -> dict[str, Any]:
    query = urllib.parse.urlencode({"version": version})
    return request_json("POST", f"/triage?{query}", payload_for(case))


records: list[dict[str, Any]] = []
total_requests = len(VERSIONS) * len(cases)
completed_requests = 0
overall_started = time.perf_counter()
print(f"Starting triage evaluation: {len(cases)} cases x {len(VERSIONS)} versions = {total_requests} requests", flush=True)

for version_index, version in enumerate(VERSIONS, start=1):
    version_started = time.perf_counter()
    print(f"Starting {version} ({version_index}/{len(VERSIONS)})", flush=True)
    for case_index, case in enumerate(cases, start=1):
        case_started = time.perf_counter()
        try:
            response = call_triage(case, version)
            error = None
            status = "ok"
        except Exception as exc:
            response = None
            error = str(exc)
            status = "error"
        records.append({"version": version, "case": case, "response": response, "error": error})
        completed_requests += 1
        percent = completed_requests / total_requests * 100 if total_requests else 100
        elapsed = time.perf_counter() - case_started
        error_summary = f" - {error[:300]}" if error else ""
        print(
            f"[{completed_requests}/{total_requests} | {percent:5.1f}%] "
            f"{version} case {case_index}/{len(cases)} {case['id']} {status} in {elapsed:.2f}s{error_summary}",
            flush=True,
        )
    print(f"{version}: completed {len(cases)} cases in {time.perf_counter() - version_started:.2f}s", flush=True)

print(f"Finished triage evaluation in {time.perf_counter() - overall_started:.2f}s", flush=True)
len(records)


## Scoring Rules

The labels in the JSONL are benchmark metadata, not service inputs.

The evaluator separates benchmark labels into:

- `scenario_tags`: scenario-shape labels, useful for coverage reporting but not part of end-to-end triage pass/fail.
- `expected_signals`: operational facts the service should emit in structured triage fields.
- `expected_preferences`: explicit customer communication preferences.
- `forbidden_signals`: structured signals the service should not emit.

For backward compatibility, older `must_detect` labels are split into those groups during scoring.

Metrics:

- `category_ok`: exact match on expected complaint category
- `severity_ok`: exact match on expected severity
- `routing_ok`: exact match on expected routing
- `must_detect_recall`: recall across expected operational signals and preferences
- `operational_signal_recall`: recall across expected operational facts only
- `preference_recall`: recall across explicit customer preferences
- `scenario_tag_recall`: coverage check for scenario-shape tags, reported separately
- `must_not_false_positive_count`: number of forbidden structured signals detected
- `metadata_accuracy`: exact match on non-null expected metadata fields
- `acknowledgement_safe`: deterministic safety check on the acknowledgement draft
- `end_to_end_ok`: category, severity, routing, operational signal, forbidden signal, metadata, and acknowledgement checks passed


In [ ]:
ALIASES: dict[str, list[str]] = {
    "incorrect_staff_information": ["incorrect staff", "wrong information", "staff member told", "misinformed"],
    "direct_debit_change_error": ["direct debit", "debit change", "payment account", "direct_debit_failure"],
    "direct_debit_failure": ["direct debit", "debit failed", "failed debit", "direct_debit_failure"],
    "missed_payment": ["missed payment", "payment bounced", "bounced", "late payment"],
    "dishonour_fee": ["dishonour fee", "dishonor fee", "$15"],
    "credit_file_concern": ["credit file", "bad credit", "credit risk", "credit_file_risk", "credit file risk"],
    "fee_waiver_request": ["waive", "waived", "fee waiver", "late fee"],
    "financial_hardship": ["financial hardship", "hardship", "cannot afford", "can't afford", "repayment support", "payment difficulty"],
    "hardship": ["financial hardship", "hardship", "cannot afford", "can't afford", "repayment support"],
    "reduced_income": ["reduced income", "pay cut", "lost shifts", "lost job", "job loss", "back at work"],
    "relationship_breakdown": ["relationship breakdown", "wife moved out", "separation", "partner moved out"],
    "dependent_children": ["dependent", "children", "kids", "daycare", "seven year old", "family_or_dependent"],
    "dependent_child": ["dependent", "child", "children", "seven year old", "family_or_dependent"],
    "sleep_distress": ["sleep", "2am", "distress", "health_or_distress"],
    "repayment_assistance_request": ["pause", "repayments smaller", "repayment support", "hardship team", "assistance"],
    "repayment_pause_request": ["pause", "repayment pause", "pause repayments", "repayment support"],
    "prefers_no_phone_contact": ["no phone", "do not want to talk on the phone", "message", "prefers_message", "prefers_no_phone_contact"],
    "responsible_lending": ["responsible lending", "affordability", "unaffordable", "should not have been approved"],
    "financial_counsellor": ["financial counsellor", "counsellor", "counselor"],
    "unaffordable_loan": ["unaffordable", "can't afford", "cannot afford", "affordability"],
    "casual_income": ["casual shifts", "casual income"],
    "job_loss": ["job loss", "lost my job", "let go"],
    "arrears": ["arrears", "behind two payments", "overdue"],
    "collections_contact": ["collections", "calls", "texts", "arrears contact"],
    "collections_distress": ["collections", "calls", "contacting me about money", "distress"],
    "AFCA_escalation_risk": ["afca", "AFCA_escalation_risk"],
    "AFCA": ["afca", "AFCA_escalation_risk"],
    "routine_service_error": ["routine service", "service_error"],
    "fraud": ["fraud", "identity theft", "unauthorised", "unauthorized"],
    "self_harm": ["self harm", "self_harm", "keep myself safe", "no way out"],
    "self_harm_signal": ["self harm", "self_harm", "keep myself safe", "no way out"],
    "immediate_safety_risk": ["immediate safety", "keep myself safe", "tonight", "no way out"],
    "duplicate_charge": ["duplicate", "charged twice", "charged me twice"],
    "refund_request": ["refund"],
    "unclear_issue": ["unclear", "vague", "insufficient information"],
    "escalation_request": ["senior", "escalation", "escalate"],
    "wrong_company_or_product": ["wrong company", "wrong product", "insurance", "do not have a loan", "not the fridge finance company"],
    "no_fundsmart_account_claimed": ["no fundsmart account", "do not have a loan", "not a fundsmart customer", "no account"],
    "minimal_action_required": ["minimal action", "redirect", "wrong company", "frontline"],
    "loan_complaint": ["loan complaint", "loan"],
    "late_fee": ["late fee"],
    "cannot_afford_repayment": ["cannot afford", "can't afford", "payment difficulty", "hardship"],
    "payment_arrangement": ["payment arrangement", "arrangement"],
    "payment_arrangement_dispute": ["payment arrangement", "arrangement", "arrangement not honoured"],
    "payment_allocation_dispute": ["payment allocation", "allocated", "wrong account", "allocation"],
    "medical_vulnerability": ["surgery", "medical", "health_or_distress"],
    "contact_preference": ["cannot take calls", "no phone", "message", "preference"],
    "payment_not_recognised": ["payment not recognised", "payment not recognized", "already paid", "pay already"],
    "app_payment_status_error": ["app", "payment status", "already paid", "payment not recognised"],
    "prefers_in_app_message": ["message me here", "in app", "prefers_message", "prefers_in_app_message"],
    "prefers_email_contact": ["email", "prefers_email_contact"],
    "document_upload_issue": ["document", "uploaded", "payslips", "upload"],
    "application_delay": ["delayed", "application delay", "refinance application"],
    "abusive_language": ["abuse", "abusive", "thieves", "useless"],
    "threat_to_property": ["threat", "property", "office", "coming down"],
    "staff_safety_risk": ["staff safety", "threat", "coming down", "regret it"],
    "abuse_threat": ["threat", "regret it", "coming down to your office"],
    "identity_theft": ["identity theft", "used my identity", "opened in my name", "identity_theft"],
    "unauthorised_loan": ["unauthorised", "unauthorized", "never applied", "opened in my name"],
    "stolen_wallet": ["stolen wallet", "wallet stolen"],
    "freeze_account_request": ["freeze", "freeze account"],
    "stop_debits_request": ["stop debit", "stop debits", "stop direct debit"],
    "credit_file_risk": ["credit file", "credit risk", "credit_file_risk"],
    "routine_payment_issue": ["routine payment", "payment issue"],
    "financial_distress": ["financial distress", "too much", "no way out", "hardship"],
    "collections_contact_distress": ["collections", "calls", "contacting me about money"],
    "threatening_language": ["threat", "regret it", "coming down to your office"],
    "collections_complaint": ["collections", "collections complaint"],
    "late_payment_dispute": ["late payment", "late mark"],
    "contradictory_information": ["contradictory", "contradiction", "did pay two days after"],
    "app_issue": ["app", "logging me out"],
    "login_issue": ["login", "logging me out"],
    "statement_request": ["statement"],
    "workaround_available": ["workaround", "send statement", "email statement"],
    "no_overdue_payment": ["not overdue", "no overdue", "not late"],
}

SCENARIO_TAG_LABELS = {
    "very_short_complaint",
    "vague_angry_rant",
    "wrong_company_or_product",
    "hidden_hardship_in_fee_complaint",
    "multi_issue_complaint",
    "esl_style_writing",
    "sarcastic_complaint",
    "sarcasm",
    "fraud_or_identity_theft",
    "self_harm_signal_scenario",
    "abusive_or_threatening_customer",
    "contradictory_complaint",
    "routine_app_bug",
    "seed_service_complaint",
    "seed_subtle_hardship",
    "seed_responsible_lending_afca",
}

PREFERENCE_LABELS = {
    "prefers_no_phone_contact",
    "prefers_in_app_message",
    "prefers_email_contact",
    "prefers_message_or_no_phone_contact",
    "contact_preference",
}


def norm_text(value: str) -> str:
    value = value.replace("_", " ").lower()
    return re.sub(r"\s+", " ", value)


def flatten_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, dict):
        return " ".join(flatten_text(v) for v in value.values())
    if isinstance(value, list):
        return " ".join(flatten_text(v) for v in value)
    return str(value)


def structured_signal_text(response: dict[str, Any]) -> str:
    triage = response.get("triage") or {}
    fields = {
        "category": triage.get("category"),
        "severity": triage.get("severity"),
        "detected_signals": triage.get("detected_signals"),
        "vulnerability_signals": triage.get("vulnerability_signals"),
        "regulatory_flags": triage.get("regulatory_flags"),
        "recommended_routing": triage.get("recommended_routing"),
        "sla_recommendation": triage.get("sla_recommendation"),
        "customer_preferences": triage.get("customer_preferences"),
    }
    return norm_text(flatten_text(fields))


def scenario_text(case: dict[str, Any]) -> str:
    return norm_text(" ".join(str(value) for value in [case.get("scenario_type"), *(case.get("scenario_tags") or [])] if value))


def alias_hit(label: str, text: str) -> bool:
    aliases = ALIASES.get(label, [label])
    candidates = aliases + [label, label.replace("_", " ")]
    return any(norm_text(candidate) in text for candidate in candidates)


def label_taxonomy(case: dict[str, Any]) -> dict[str, list[str]]:
    must_detect = case.get("must_detect") or []
    explicit_scenario_tags = case.get("scenario_tags") or []
    explicit_expected_signals = case.get("expected_signals") or []
    explicit_expected_preferences = case.get("expected_preferences") or []
    explicit_forbidden = case.get("forbidden_signals") or []
    explicit_customer_preferences = case.get("customer_preferences") or []

    scenario_tags = explicit_scenario_tags or [label for label in must_detect if label in SCENARIO_TAG_LABELS]
    expected_preferences = explicit_expected_preferences or explicit_customer_preferences or [
        label for label in must_detect if label in PREFERENCE_LABELS or label.startswith("prefers_")
    ]
    expected_signals = explicit_expected_signals or [
        label for label in must_detect if label not in set(scenario_tags) and label not in set(expected_preferences)
    ]
    forbidden_signals = explicit_forbidden or case.get("must_not_detect") or []
    return {
        "scenario_tags": scenario_tags,
        "expected_signals": expected_signals,
        "expected_preferences": expected_preferences,
        "forbidden_signals": forbidden_signals,
    }


def metadata_score(expected: dict[str, Any], actual: dict[str, Any]) -> tuple[float, list[str]]:
    checked = 0
    correct = 0
    mismatches: list[str] = []
    for key, expected_value in (expected or {}).items():
        if expected_value is None:
            continue
        checked += 1
        actual_value = (actual or {}).get(key)
        if norm_text(str(actual_value or "")) == norm_text(str(expected_value)):
            correct += 1
        else:
            mismatches.append(f"{key}: expected={expected_value!r}, actual={actual_value!r}")
    return (correct / checked if checked else 1.0), mismatches


def acknowledgement_safety(acknowledgement: str) -> tuple[bool, list[str]]:
    lower = norm_text(acknowledgement or "")
    errors: list[str] = []
    if len((acknowledgement or "").strip()) < 40:
        errors.append("too_short")
    if "received" not in lower:
        errors.append("does_not_confirm_receipt")
    if "sorry" not in lower and "understand" not in lower:
        errors.append("not_empathetic")
    banned = ["we will refund", "we accept liability", "we are liable", "we will waive"]
    if any(phrase in lower for phrase in banned):
        errors.append("promises_outcome_or_admits_liability")
    return not errors, errors


def score_record(record: dict[str, Any]) -> dict[str, Any]:
    case = record["case"]
    response = record["response"]
    taxonomy = label_taxonomy(case)
    expected_signals = taxonomy["expected_signals"]
    expected_preferences = taxonomy["expected_preferences"]
    expected_operational = [*expected_signals, *expected_preferences]
    forbidden_signals = taxonomy["forbidden_signals"]
    if response is None:
        return {
            "version": record["version"],
            "id": case["id"],
            "scenario_type": case.get("scenario_type"),
            "source": case.get("source"),
            "error": record["error"],
            "category_ok": False,
            "severity_ok": False,
            "routing_ok": False,
            "must_detect_recall": 0.0,
            "operational_signal_recall": 0.0,
            "scenario_tag_recall": 0.0,
            "preference_recall": 0.0,
            "must_not_false_positive_count": len(forbidden_signals),
            "metadata_accuracy": 0.0,
            "acknowledgement_safe": False,
            "end_to_end_ok": False,
            "missing_must_detect": expected_operational,
            "missing_expected_signals": expected_signals,
            "missing_expected_preferences": expected_preferences,
            "missing_scenario_tags": taxonomy["scenario_tags"],
            "unexpected_must_not": forbidden_signals,
            "metadata_mismatches": [],
            "acknowledgement_errors": ["request_failed"],
        }

    triage = response.get("triage") or {}
    signal_text = structured_signal_text(response)
    scenario_hit_text = signal_text + " " + scenario_text(case)
    found_signals = [label for label in expected_signals if alias_hit(label, signal_text)]
    missing_signals = [label for label in expected_signals if label not in found_signals]
    found_preferences = [label for label in expected_preferences if alias_hit(label, signal_text)]
    missing_preferences = [label for label in expected_preferences if label not in found_preferences]
    found_scenario_tags = [label for label in taxonomy["scenario_tags"] if alias_hit(label, scenario_hit_text)]
    missing_scenario_tags = [label for label in taxonomy["scenario_tags"] if label not in found_scenario_tags]
    unexpected = [label for label in forbidden_signals if alias_hit(label, signal_text)]
    metadata_accuracy, metadata_mismatches = metadata_score(
        case.get("metadata") or {},
        triage.get("extracted_metadata") or {},
    )
    ack_safe, ack_errors = acknowledgement_safety(response.get("acknowledgement_draft") or "")
    category_ok = triage.get("category") == case.get("expected_category")
    severity_ok = triage.get("severity") == case.get("expected_severity")
    routing_ok = triage.get("recommended_routing") == case.get("expected_routing")
    operational_total = len(expected_signals) + len(expected_preferences)
    operational_found = len(found_signals) + len(found_preferences)
    operational_recall = operational_found / operational_total if operational_total else 1.0
    signal_recall = len(found_signals) / len(expected_signals) if expected_signals else 1.0
    preference_recall = len(found_preferences) / len(expected_preferences) if expected_preferences else 1.0
    scenario_tag_recall = len(found_scenario_tags) / len(taxonomy["scenario_tags"]) if taxonomy["scenario_tags"] else 1.0
    end_to_end_ok = all(
        [
            category_ok,
            severity_ok,
            routing_ok,
            operational_recall == 1.0,
            not unexpected,
            metadata_accuracy == 1.0,
            ack_safe,
        ]
    )
    return {
        "version": record["version"],
        "id": case["id"],
        "scenario_type": case.get("scenario_type"),
        "source": case.get("source"),
        "expected_category": case.get("expected_category"),
        "actual_category": triage.get("category"),
        "expected_severity": case.get("expected_severity"),
        "actual_severity": triage.get("severity"),
        "expected_routing": case.get("expected_routing"),
        "actual_routing": triage.get("recommended_routing"),
        "category_ok": category_ok,
        "severity_ok": severity_ok,
        "routing_ok": routing_ok,
        "must_detect_recall": operational_recall,
        "operational_signal_recall": signal_recall,
        "scenario_tag_recall": scenario_tag_recall,
        "preference_recall": preference_recall,
        "must_not_false_positive_count": len(unexpected),
        "metadata_accuracy": metadata_accuracy,
        "acknowledgement_safe": ack_safe,
        "end_to_end_ok": end_to_end_ok,
        "missing_must_detect": [*missing_signals, *missing_preferences],
        "missing_expected_signals": missing_signals,
        "missing_expected_preferences": missing_preferences,
        "missing_scenario_tags": missing_scenario_tags,
        "unexpected_must_not": unexpected,
        "metadata_mismatches": metadata_mismatches,
        "acknowledgement_errors": ack_errors,
        "detected_signals": triage.get("detected_signals") or [],
        "vulnerability_signals": triage.get("vulnerability_signals") or [],
        "regulatory_flags": triage.get("regulatory_flags") or [],
        "customer_preferences": triage.get("customer_preferences") or [],
        "error": record["error"],
        "latency_seconds": response.get("latency_seconds"),
    }

scores = pd.DataFrame(score_record(record) for record in records)
scores.head()


In [ ]:
metric_columns = [
    "category_ok",
    "severity_ok",
    "routing_ok",
    "must_detect_recall",
    "operational_signal_recall",
    "preference_recall",
    "scenario_tag_recall",
    "metadata_accuracy",
    "acknowledgement_safe",
    "end_to_end_ok",
]

summary = (
    scores.assign(no_forbidden_signal=scores["must_not_false_positive_count"].eq(0))
    .groupby("version")
    .agg(
        cases=("id", "count"),
        category_accuracy=("category_ok", "mean"),
        severity_accuracy=("severity_ok", "mean"),
        routing_accuracy=("routing_ok", "mean"),
        must_detect_recall=("must_detect_recall", "mean"),
        operational_signal_recall=("operational_signal_recall", "mean"),
        preference_recall=("preference_recall", "mean"),
        scenario_tag_recall=("scenario_tag_recall", "mean"),
        no_forbidden_signal_rate=("no_forbidden_signal", "mean"),
        metadata_accuracy=("metadata_accuracy", "mean"),
        acknowledgement_safety_rate=("acknowledgement_safe", "mean"),
        end_to_end_pass_rate=("end_to_end_ok", "mean"),
        mean_latency_seconds=("latency_seconds", "mean"),
    )
    .round(3)
)

summary


In [ ]:
pivot = summary.drop(columns=["cases"]).T
if set(VERSIONS).issubset(pivot.columns):
    pivot["v2_minus_v1"] = pivot["v2"] - pivot["v1"]
pivot

## Failure Analysis

This table is the useful part for iteration. It shows where each version missed labels, over-detected forbidden signals, failed metadata extraction, or produced an unsafe acknowledgement.

In [ ]:
failure_columns = [
    "version",
    "id",
    "scenario_type",
    "expected_category",
    "actual_category",
    "expected_severity",
    "actual_severity",
    "expected_routing",
    "actual_routing",
    "missing_expected_signals",
    "missing_expected_preferences",
    "missing_scenario_tags",
    "unexpected_must_not",
    "metadata_mismatches",
    "acknowledgement_errors",
    "detected_signals",
    "vulnerability_signals",
    "regulatory_flags",
    "customer_preferences",
    "error",
]

failures = scores[
    (~scores["end_to_end_ok"])
    | scores["error"].notna()
][failure_columns]

failures


In [ ]:
by_scenario = (
    scores.assign(no_forbidden_signal=scores["must_not_false_positive_count"].eq(0))
    .groupby(["version", "scenario_type"])
    .agg(
        category_accuracy=("category_ok", "mean"),
        severity_accuracy=("severity_ok", "mean"),
        routing_accuracy=("routing_ok", "mean"),
        must_detect_recall=("must_detect_recall", "mean"),
        operational_signal_recall=("operational_signal_recall", "mean"),
        preference_recall=("preference_recall", "mean"),
        scenario_tag_recall=("scenario_tag_recall", "mean"),
        no_forbidden_signal_rate=("no_forbidden_signal", "mean"),
        metadata_accuracy=("metadata_accuracy", "mean"),
        acknowledgement_safety_rate=("acknowledgement_safe", "mean"),
        end_to_end_pass_rate=("end_to_end_ok", "mean"),
    )
    .round(3)
)

by_scenario


## Optional Ragas Acknowledgement Evaluation

Ragas is used only for the free-text acknowledgement. The deterministic gold-label metrics above remain the source of truth for category, severity, routing, metadata, and signal detection.

Set `RUN_RAGAS_EVAL=true` before running the notebook to enable this section. The judge model comes from `RAGAS_LLM_MODEL`, then `UTILITY_LLM_MODEL`, then `REASONING_LLM_MODEL`.


In [ ]:
RUN_RAGAS_EVAL = str(RUN_RAGAS_EVAL_PARAM).strip().lower() == "true"
RAGAS_LLM_MODEL = (
    os.getenv("RAGAS_LLM_MODEL")
    or os.getenv("UTILITY_LLM_MODEL")
    or os.getenv("REASONING_LLM_MODEL")
)
RAGAS_SAMPLE_LIMIT = int(os.getenv("RAGAS_SAMPLE_LIMIT", "0"))

ragas_scores = pd.DataFrame()
ragas_error: str | None = None

def acknowledgement_reference(case: dict[str, Any]) -> str:
    return "\n".join(
        [
            "Evaluate only the acknowledgement draft, not the structured triage labels.",
            "A strong acknowledgement should confirm receipt, be empathetic but neutral, stay grounded in the complaint, avoid admitting liability, avoid promising a refund or credit-file correction, and avoid giving legal or financial advice.",
            "It should acknowledge urgent vulnerability or regulatory context when present and should not contradict the expected benchmark labels.",
            f"Expected category: {case.get('expected_category')}",
            f"Expected severity: {case.get('expected_severity')}",
            f"Expected routing: {case.get('expected_routing')}",
            f"Must-detect signals: {case.get('must_detect') or []}",
            f"Must-not-detect signals: {case.get('must_not_detect') or []}",
        ]
    )

def acknowledgement_response_text(record: dict[str, Any]) -> str:
    response = record.get("response") or {}
    triage = response.get("triage") or {}
    return "\n".join(
        [
            f"Acknowledgement draft:\n{response.get('acknowledgement_draft') or ''}",
            "",
            "Structured context produced by the service:",
            json.dumps(
                {
                    "category": triage.get("category"),
                    "severity": triage.get("severity"),
                    "recommended_routing": triage.get("recommended_routing"),
                    "detected_signals": triage.get("detected_signals"),
                    "vulnerability_signals": triage.get("vulnerability_signals"),
                    "regulatory_flags": triage.get("regulatory_flags"),
                    "customer_preferences": triage.get("customer_preferences"),
                },
                ensure_ascii=False,
            ),
        ]
    )

if RUN_RAGAS_EVAL:
    if not RAGAS_LLM_MODEL:
        ragas_error = "RUN_RAGAS_EVAL=true but no RAGAS_LLM_MODEL, UTILITY_LLM_MODEL, or REASONING_LLM_MODEL is set."
        print(ragas_error)
    else:
        try:
            from langchain_openai import ChatOpenAI
            from ragas import EvaluationDataset, SingleTurnSample, evaluate
            from ragas.llms import LangchainLLMWrapper
            from ragas.metrics import AspectCritic, MetricType

            successful_records = [record for record in records if record.get("response") is not None]
            if RAGAS_SAMPLE_LIMIT > 0:
                successful_records = successful_records[:RAGAS_SAMPLE_LIMIT]

            ragas_metadata = [
                {
                    "version": record["version"],
                    "id": record["case"]["id"],
                    "scenario_type": record["case"].get("scenario_type"),
                }
                for record in successful_records
            ]
            ragas_dataset = EvaluationDataset(
                samples=[
                    SingleTurnSample(
                        user_input=record["case"]["complaint_document"],
                        response=acknowledgement_response_text(record),
                        reference=acknowledgement_reference(record["case"]),
                    )
                    for record in successful_records
                ]
            )
            required = {MetricType.SINGLE_TURN: {"user_input", "response", "reference"}}
            ragas_llm = LangchainLLMWrapper(ChatOpenAI(model=RAGAS_LLM_MODEL, temperature=0))
            ragas_metrics = [
                AspectCritic(
                    name="acknowledgement_groundedness",
                    definition=(
                        "Return 1 only if the acknowledgement is faithful to the complaint and structured context, "
                        "does not invent facts, does not assert the business made an error unless already established, "
                        "and does not promise an outcome. Otherwise return 0."
                    ),
                    llm=ragas_llm,
                    required_columns=required,
                    strictness=3,
                ),
                AspectCritic(
                    name="acknowledgement_coherence",
                    definition=(
                        "Return 1 only if the acknowledgement is clear, internally consistent, relevant to the complaint, "
                        "and aligned with the case type and routing context. Otherwise return 0."
                    ),
                    llm=ragas_llm,
                    required_columns=required,
                    strictness=3,
                ),
                AspectCritic(
                    name="acknowledgement_safety_compliance",
                    definition=(
                        "Return 1 only if the acknowledgement is safe for a financial complaints workflow: it confirms receipt, "
                        "uses neutral empathetic language, avoids liability admission, avoids refund or credit-file promises, "
                        "does not give legal or financial advice, and handles vulnerability, self-harm, hardship, fraud, AFCA, "
                        "or responsible-lending context with appropriate escalation language. Otherwise return 0."
                    ),
                    llm=ragas_llm,
                    required_columns=required,
                    strictness=3,
                ),
            ]
            ragas_result = evaluate(
                dataset=ragas_dataset,
                metrics=ragas_metrics,
                raise_exceptions=False,
                show_progress=True,
            )
            ragas_scores = ragas_result.to_pandas()
            ragas_scores = pd.concat([pd.DataFrame(ragas_metadata), ragas_scores], axis=1)
            display(ragas_scores)
        except Exception as exc:
            ragas_error = str(exc)
            print(f"Ragas acknowledgement evaluation failed: {ragas_error}")
else:
    print("Ragas acknowledgement evaluation skipped. Set RUN_RAGAS_EVAL=true to enable it.")


## Results Explanation

This section turns the metric tables into presentation-ready interpretation. It includes deterministic triage metrics and, when enabled, Ragas acknowledgement-quality results.


In [ ]:
from IPython.display import Markdown, display


def pct(value: float | int | None) -> str:
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value) * 100:.1f}%"


def number(value: float | int | None) -> str:
    if value is None or pd.isna(value):
        return "n/a"
    return f"{float(value):.3f}"


def metric_value(version: str, metric: str) -> float | None:
    if version not in summary.index or metric not in summary.columns:
        return None
    return float(summary.loc[version, metric])


versions_available = list(summary.index)
lines: list[str] = ["### Evaluation Result Interpretation", ""]
lines.append(f"The run evaluated **{int(summary['cases'].max()) if 'cases' in summary else len(cases)} cases** across **{len(versions_available)} prompt versions**.")
lines.append("")

if {"v1", "v2"}.issubset(summary.index):
    comparisons = [
        ("Category accuracy", "category_accuracy", True),
        ("Severity accuracy", "severity_accuracy", True),
        ("Routing accuracy", "routing_accuracy", True),
        ("Must-detect recall", "must_detect_recall", True),
        ("Operational signal recall", "operational_signal_recall", True),
        ("Preference recall", "preference_recall", True),
        ("Scenario tag recall", "scenario_tag_recall", True),
        ("No forbidden-signal rate", "no_forbidden_signal_rate", True),
        ("Metadata accuracy", "metadata_accuracy", True),
        ("Acknowledgement safety rate", "acknowledgement_safety_rate", True),
        ("End-to-end pass rate", "end_to_end_pass_rate", True),
        ("Mean latency seconds", "mean_latency_seconds", False),
    ]
    lines.append("| Metric | v1 | v2 | Change |")
    lines.append("|---|---:|---:|---:|")
    for label, metric, as_percent in comparisons:
        v1 = metric_value("v1", metric)
        v2 = metric_value("v2", metric)
        delta = None if v1 is None or v2 is None else v2 - v1
        if as_percent:
            lines.append(f"| {label} | {pct(v1)} | {pct(v2)} | {pct(delta)} |")
        else:
            lines.append(f"| {label} | {number(v1)} | {number(v2)} | {number(delta)} |")
    lines.append("")

    core_metrics = ["category_accuracy", "severity_accuracy", "routing_accuracy", "must_detect_recall", "operational_signal_recall", "no_forbidden_signal_rate", "metadata_accuracy", "acknowledgement_safety_rate", "end_to_end_pass_rate"]
    improved_core = []
    regressed_core = []
    unchanged_core = []
    for metric in core_metrics:
        v1 = metric_value("v1", metric)
        v2 = metric_value("v2", metric)
        if v1 is None or v2 is None:
            continue
        if v2 > v1:
            improved_core.append(metric)
        elif v2 < v1:
            regressed_core.append(metric)
        else:
            unchanged_core.append(metric)

    lines.append("### Overall Judgement")
    lines.append("")
    if regressed_core and improved_core:
        lines.append("v2 is **not a clean overall improvement** in this run. It improved some safety/preference metrics, but it regressed on some core triage metrics.")
    elif regressed_core and not improved_core:
        lines.append("v2 regressed on the measured deterministic triage metrics in this run.")
    elif improved_core and not regressed_core:
        lines.append("v2 improved on the measured deterministic triage metrics in this run.")
    else:
        lines.append("v1 and v2 are broadly tied on the measured deterministic triage metrics in this run.")
    lines.append(f"- Improved core metrics: {', '.join(improved_core) if improved_core else 'none'}.")
    lines.append(f"- Regressed core metrics: {', '.join(regressed_core) if regressed_core else 'none'}.")
    lines.append(f"- Unchanged core metrics: {', '.join(unchanged_core) if unchanged_core else 'none'}.")
    lines.append("")

    lines.append("### What Improved")
    lines.append("")
    improvements: list[str] = []
    for label, metric, as_percent in comparisons:
        v1 = metric_value("v1", metric)
        v2 = metric_value("v2", metric)
        if v1 is None or v2 is None:
            continue
        delta = v2 - v1
        better = delta > 0 if metric != "mean_latency_seconds" else delta < 0
        if better:
            improvements.append(
                f"- **{label}** improved from {pct(v1) if as_percent else number(v1)} to {pct(v2) if as_percent else number(v2)}."
            )
    lines.extend(improvements or ["- No major metric improved in this run."])
    lines.append("")

    lines.append("### What Regressed Or Still Needs Work")
    lines.append("")
    regressions: list[str] = []
    for label, metric, as_percent in comparisons:
        v1 = metric_value("v1", metric)
        v2 = metric_value("v2", metric)
        if v1 is None or v2 is None:
            continue
        delta = v2 - v1
        worse = delta < 0 if metric != "mean_latency_seconds" else delta > 0
        if worse:
            regressions.append(
                f"- **{label}** moved from {pct(v1) if as_percent else number(v1)} to {pct(v2) if as_percent else number(v2)}."
            )
    if metric_value("v2", "end_to_end_pass_rate") == 0:
        regressions.append("- **End-to-end pass rate is still 0%**, because the benchmark requires every category, severity, routing, operational signal, preference, forbidden-signal, metadata, and acknowledgement check to pass in the same case.")
    lines.extend(regressions or ["- No major regression was measured in this run."])
    lines.append("")

    lines.append("### Triage Interpretation")
    lines.append("")
    lines.append("- Category and routing accuracy are useful headline metrics, but they do not tell the full story. The stricter signal checks reveal whether the model captured the exact operational facts required by the benchmark.")
    lines.append("- `must_detect_recall` now means recall across operational signals plus explicit customer preferences. Scenario-shape tags are reported separately as `scenario_tag_recall` and do not drive end-to-end pass/fail.")
    lines.append("- The forbidden-signal rate is now based on structured fields (`detected_signals`, `vulnerability_signals`, `regulatory_flags`, and `customer_preferences`) instead of broad matching across all generated prose. That makes the metric less vulnerable to negated text such as `no evidence of hardship`.")
    lines.append("")

    lines.append("### Ragas Acknowledgement Evaluation")
    lines.append("")
    if "ragas_scores" in globals() and isinstance(ragas_scores, pd.DataFrame) and not ragas_scores.empty:
        ragas_metric_columns = [
            col for col in ragas_scores.columns
            if col not in {"version", "id", "scenario_type"} and pd.api.types.is_numeric_dtype(ragas_scores[col])
        ]
        if ragas_metric_columns:
            ragas_summary = ragas_scores.groupby("version")[ragas_metric_columns].mean().round(3)
            lines.append("Ragas was enabled for acknowledgement-only judging. It should be read separately from deterministic triage scoring.")
            lines.append("")
            lines.append("| Ragas metric | " + " | ".join(ragas_summary.index.astype(str)) + " |")
            lines.append("|---" + "|---:" * len(ragas_summary.index) + "|")
            for metric in ragas_metric_columns:
                values = [pct(ragas_summary.loc[version, metric]) for version in ragas_summary.index]
                lines.append(f"| {metric} | " + " | ".join(values) + " |")
            lines.append("")
            if {"v1", "v2"}.issubset(ragas_summary.index):
                improved = [metric for metric in ragas_metric_columns if ragas_summary.loc["v2", metric] > ragas_summary.loc["v1", metric]]
                regressed = [metric for metric in ragas_metric_columns if ragas_summary.loc["v2", metric] < ragas_summary.loc["v1", metric]]
                tied = [metric for metric in ragas_metric_columns if ragas_summary.loc["v2", metric] == ragas_summary.loc["v1", metric]]
                lines.append(f"- Ragas-improved acknowledgement metrics: {', '.join(improved) if improved else 'none'}.")
                lines.append(f"- Ragas-regressed acknowledgement metrics: {', '.join(regressed) if regressed else 'none'}.")
                lines.append(f"- Ragas-tied acknowledgement metrics: {', '.join(tied) if tied else 'none'}.")
        else:
            lines.append("Ragas produced rows, but no numeric metric columns were found to summarize.")
    elif "ragas_error" in globals() and ragas_error:
        lines.append(f"Ragas was requested but failed: `{ragas_error}`")
    else:
        lines.append("Ragas was not run for this notebook execution. Set `RUN_RAGAS_EVAL=true` when running Papermill to include acknowledgement groundedness/coherence/safety judge results here.")
    lines.append("")

    lines.append("### Best Concrete Failure Examples")
    lines.append("")
    interesting_failures = scores[
        (~scores["category_ok"]) | (~scores["severity_ok"]) | (~scores["routing_ok"])
    ][
        [
            "version",
            "id",
            "expected_category",
            "actual_category",
            "expected_severity",
            "actual_severity",
            "expected_routing",
            "actual_routing",
            "missing_expected_signals",
            "unexpected_must_not",
        ]
    ]
    if interesting_failures.empty:
        lines.append("- No category, severity, or routing failures were found. Remaining failures are signal, metadata, or acknowledgement checks.")
    else:
        for row in interesting_failures.head(5).itertuples(index=False):
            lines.append(
                f"- **{row.version} / {row.id}**: category `{row.expected_category}` -> `{row.actual_category}`, "
                f"severity `{row.expected_severity}` -> `{row.actual_severity}`, routing `{row.expected_routing}` -> `{row.actual_routing}`."
            )
    lines.append("")

    lines.append("### Recommended Next Iteration")
    lines.append("")
    lines.append("1. Keep using `detected_signals` as the primary surface for operational fact scoring, and keep broad generated prose out of the signal metrics.")
    lines.append("2. Calibrate v2 severity rules further. This run shows v2 can over-escalate some cases while improving forbidden-signal suppression.")
    lines.append("3. Review category regressions case-by-case before changing prompts. If v1 beats v2 on category in a batch, inspect whether v2 is being too fee-focused or risk-focused.")
    lines.append("4. Keep Ragas acknowledgement metrics separate from deterministic triage metrics. Acknowledgement quality can improve even when triage classification regresses.")
else:
    lines.append("The summary does not contain both `v1` and `v2`, so version comparison could not be generated.")

display(Markdown("\n".join(lines)))


## Save Evaluation Artefacts

The JSON output can be used in the presentation to show the measured v1/v2 result and the concrete failure cases that informed iteration.

In [ ]:
output_dir = EVALUATION_OUTPUT_DIR_PATH
output_dir.mkdir(parents=True, exist_ok=True)

summary_path = output_dir / "summary.json"
scores_path = output_dir / "case_scores.jsonl"
failures_path = output_dir / "failures.jsonl"
ragas_path = output_dir / "ragas_acknowledgement_scores.jsonl"

summary_path.write_text(summary.reset_index().to_json(orient="records", indent=2))
scores_path.write_text("\n".join(json.dumps(row, default=str) for row in scores.to_dict(orient="records")) + "\n")
failures_path.write_text("\n".join(json.dumps(row, default=str) for row in failures.to_dict(orient="records")) + "\n")
if "ragas_scores" in globals() and not ragas_scores.empty:
    ragas_path.write_text("\n".join(json.dumps(row, default=str) for row in ragas_scores.to_dict(orient="records")) + "\n")

print(f"Wrote {summary_path}")
print(f"Wrote {scores_path}")
print(f"Wrote {failures_path}")
if "ragas_scores" in globals() and not ragas_scores.empty:
    print(f"Wrote {ragas_path}")


## Presentation Notes Template

Use the measured output above to fill this in:

- **v1 baseline:** basic prompt, weaker explicit definitions, fewer safety/routing instructions.
- **v1 failure modes:** use the failure table, especially missed vulnerability/regulatory signals and unsafe acknowledgement issues.
- **v2 changes:** stronger schema expectations, explicit metadata extraction, explicit vulnerability/regulatory labels, acknowledgement constraints, deterministic routing guardrails.
- **Measured result:** use `summary` and `pivot` above.
- **Next iteration:** expand labelled cases, add human-review rubric for acknowledgement quality, and track false positives for hardship/responsible-lending escalation.